# SoSe26 Case Study – Group 38

**Participants:** *(add names)*

## Table of Contents

1. [Importing the data](#1-importing-the-data)
2. [Data preparation](#2-data-preparation)
3. [Creation of the final dataset](#3-creation-of-the-final-dataset)
4. [Evaluation and Result](#4-evaluation-and-result)

---

## Task

We recommend the **most popular vehicle** for management by choosing the most
popular **component type** in each category **K1–K7**, measured by **KBA
registrations** (not production). We also look at each year to see whether
winners are stable or change like a fashion.

**How we count a component.** An ID such as `K1BE1-101-1011-7` is one physical
unit. `K1BE1-101-1011` and `K1BE1-104-1041` are the same **type** `K1BE1` from
two plants. We compare types:

- K1: `K1BE1`, `K1BE2`, `K1DI1`, `K1DI2`
- K2: `K2ST1`, `K2ST2`, `K2LE1`, `K2LE2`
- K3: `K3SG1`, `K3SG2`, `K3AG1`, `K3AG2`
- K4–K7: only one type each (`K4`, `K5`, `K6`, `K7`)

Plant and serial number are not used in the ranking.


## 1. Importing the data

### Interpretation

Management wants a future vehicle built from the components customers already
chose. Popularity = how many **registered** vehicles contain that type.

A finished car has four installed components: engine (K1), seats (K2), gearbox
(K3), body (K4 **or** K5 **or** K6 **or** K7). K4–K7 cannot be combined.

### Data-selection strategy

We load only what the question needs (lecture: do not import unnecessary
tables — RAM):

| Use | Folder / files | Why |
|-----|----------------|-----|
| Yes | `Fahrzeug/Bestandteile_Fahrzeuge_*` | Which K1–K7 types sit in which vehicle |
| Yes | `Zulassungen/Zulassungen_alle_Fahrzeuge.csv` | Registration date → year and count |
| No | `Einzelteil` | Parts inside components; not needed to rank K1–K7 |
| No | `Komponente` production tables | Plant/defect detail; we rank type, not plant |
| No | `Logistikverzug` | General Task 1 only |

Join key: `Bestandteile.ID_Fahrzeug` = `Zulassungen.IDNummer`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_DIR = Path("Data")
FINAL_CSV = DATA_DIR / "SoSe26_Case_Study_finalData_Group_38.csv"

BOM_FILES = [
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", "OEM1", "Typ11"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", "OEM1", "Typ12"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv", "OEM2", "Typ21"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv", "OEM2", "Typ22"),
]

SLOTS = {
    "ID_Motor": "K1",
    "ID_Sitze": "K2",
    "ID_Schaltung": "K3",
    "ID_Karosserie": "body",
}

TYPE_LABEL = {
    "K1BE1": "Petrol engine (OEM1)",
    "K1BE2": "Petrol engine (OEM2)",
    "K1DI1": "Diesel engine (OEM1)",
    "K1DI2": "Diesel engine (OEM2)",
    "K2ST1": "Fabric seats (OEM1)",
    "K2ST2": "Fabric seats (OEM2)",
    "K2LE1": "Leather seats (OEM1)",
    "K2LE2": "Leather seats (OEM2)",
    "K3SG1": "Manual gearbox (OEM1)",
    "K3SG2": "Manual gearbox (OEM2)",
    "K3AG1": "Automatic gearbox (OEM1)",
    "K3AG2": "Automatic gearbox (OEM2)",
    "K4": "Body (OEM1 Typ11)",
    "K5": "Body (OEM1 Typ12)",
    "K6": "Body (OEM2 Typ21)",
    "K7": "Body (OEM2 Typ22)",
}

PLOTLY_BLUES = ["#2F5F7A", "#5BA4CF", "#8FCBE8", "#C5E4F3"]
CATEGORIES = ["K1", "K2", "K3", "K4", "K5", "K6", "K7"]


def read_semicolon_csv(path, **kwargs):
    """Original tables are semicolon-separated."""
    return pd.read_csv(path, sep=";", **kwargs)


def component_type(series: pd.Series) -> pd.Series:
    """K1BE1-101-1011-7 -> K1BE1"""
    return series.astype(str).str.split("-", n=1).str[0]


In [ ]:
# Confirm relative paths before loading (examiners drop originals into Data/)
for rel, oem, vtype in BOM_FILES:
    path = DATA_DIR / rel
    print(f"{'OK' if path.exists() else 'MISSING':7}  {oem} {vtype:5}  {path}")

zul_path = DATA_DIR / "Zulassungen" / "Zulassungen_alle_Fahrzeuge.csv"
print(f"{'OK' if zul_path.exists() else 'MISSING':7}  registrations  {zul_path}")


### Load vehicle parts lists and inspect structure

Each row is one vehicle. Four ID columns are the installed engine, seats,
gearbox and body. We keep only those columns plus a label for brand and model.


In [ ]:
bom_frames = []
for rel, oem, vtype in BOM_FILES:
    path = DATA_DIR / rel
    df = read_semicolon_csv(
        path,
        usecols=["ID_Fahrzeug", "ID_Karosserie", "ID_Schaltung", "ID_Sitze", "ID_Motor"],
        dtype=str,
    )
    df["oem"] = oem
    df["vehicle_type"] = vtype
    print(
        f"{oem} {vtype}: {len(df):,} rows, "
        f"duplicate vehicle IDs={df['ID_Fahrzeug'].duplicated().sum()}, "
        f"missing cells={int(df.isna().sum().sum())}"
    )
    bom_frames.append(df)

bom = pd.concat(bom_frames, ignore_index=True)
print(f"\nTotal vehicles in parts lists: {len(bom):,}  unique IDs: {bom['ID_Fahrzeug'].nunique():,}")
bom.head()


ID pattern (from the case-study PDF): designation – manufacturer – plant –
sequential number. We keep only the designation (component type).


In [ ]:
sample = bom[["ID_Fahrzeug", "ID_Motor", "ID_Sitze", "ID_Schaltung", "ID_Karosserie"]].head(5).copy()
sample["engine_type"] = component_type(sample["ID_Motor"])
sample["seat_type"] = component_type(sample["ID_Sitze"])
sample["gearbox_type"] = component_type(sample["ID_Schaltung"])
sample["body_type"] = component_type(sample["ID_Karosserie"])
sample


### Load registrations

KBA table: vehicle ID, municipality, registration date. Popularity uses the
date (year). Municipality is not required for the type ranking.


In [ ]:
zul = read_semicolon_csv(zul_path, usecols=["IDNummer", "Gemeinden", "Zulassung"])
print(zul.dtypes)
print(f"rows={len(zul):,}  duplicate IDs={zul['IDNummer'].duplicated().sum()}  missing={zul.isna().sum().to_dict()}")
zul["Zulassung"] = pd.to_datetime(zul["Zulassung"], errors="coerce")
zul["year"] = zul["Zulassung"].dt.year.astype("Int64")
print("registration years:", sorted(zul["year"].dropna().unique().tolist()))
print("dates with parse problems:", int(zul["year"].isna().sum()))

year_counts = zul["year"].value_counts().sort_index()
fig = px.bar(
    x=year_counts.index.astype(int),
    y=year_counts.values,
    title="Registered vehicles per year",
    labels={"x": "Registration year", "y": "Vehicles"},
    color_discrete_sequence=["#5BA4CF"],
)
fig.update_layout(font_family="Source Sans Pro", plot_bgcolor="white", paper_bgcolor="white")
fig.show()
zul.head()


## 2. Data preparation

Checks before transforming: types, missing values, duplicates, join success.
Then parse the component type and reshape to tidy (long) form: one row per
vehicle × category.


In [ ]:
print("Parts-list dtypes:\n", bom.dtypes)
print("\nRegistration dtypes after parse:\n", zul.dtypes)

merged = bom.merge(
    zul[["IDNummer", "year"]],
    left_on="ID_Fahrzeug",
    right_on="IDNummer",
    how="left",
    indicator=True,
)
print("\nMerge result (left = parts list, right = registrations):")
print(merged["_merge"].value_counts())
match_rate = (merged["_merge"] == "both").mean()
print(f"Match rate: {match_rate:.1%}")
merged = merged.drop(columns=["_merge"])


In [ ]:
long_parts = []
for col, default_cat in SLOTS.items():
    tmp = merged[["year", "oem", "vehicle_type", col]].copy()
    tmp = tmp.rename(columns={col: "component_id"})
    tmp["component_type"] = component_type(tmp["component_id"])
    tmp["category"] = tmp["component_type"] if default_cat == "body" else default_cat
    long_parts.append(tmp.drop(columns=["component_id"]))

long_df = pd.concat(long_parts, ignore_index=True)
long_df = long_df.dropna(subset=["year", "component_type", "category"])
long_df["year"] = long_df["year"].astype(int)
print(f"Tidy rows (vehicle × category): {len(long_df):,}")
print("Types found per category:")
print(long_df.groupby("category")["component_type"].nunique().reindex(CATEGORIES))
long_df.head()


## 3. Creation of the final dataset

The app may use **only this file**. We store yearly registration counts by
OEM, vehicle type, category and component type — small enough to ship, complete
enough for overall totals, trends and filters.


In [ ]:
final = (
    long_df.groupby(
        ["year", "oem", "vehicle_type", "category", "component_type"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n_registrations"})
)
final["label"] = final["component_type"].map(TYPE_LABEL)
unknown = final[final["label"].isna()]
if not unknown.empty:
    raise ValueError(f"Unlabelled component types: {unknown['component_type'].unique()}")

final.to_csv(FINAL_CSV, index=False)
print(f"Wrote {FINAL_CSV}  ({len(final)} rows)")
final.head(10)


## 4. Evaluation and Result

Overall winner in a category = type with the highest `n_registrations` across
all years. Ties are reported as joint winners. Yearly line charts show whether
that ranking is stable.


In [ ]:
overall = (
    final.groupby(["category", "component_type", "label"], as_index=False)["n_registrations"]
    .sum()
    .sort_values(["category", "n_registrations"], ascending=[True, False])
)

winner_rows = []
print("Overall ranking by category\n")
for cat in CATEGORIES:
    sub = overall[overall["category"] == cat]
    print(sub.to_string(index=False))
    top = sub["n_registrations"].max()
    winner_rows.append(sub[sub["n_registrations"] == top])
    print()

winners = pd.concat(winner_rows, ignore_index=True)
print("Winners (including ties):")
winners


In [ ]:
def bar_types(data, category, title):
    fig = px.bar(
        data[data["category"] == category],
        x="component_type",
        y="n_registrations",
        color="label",
        title=title,
        labels={
            "component_type": "Component type",
            "n_registrations": "Registered vehicles",
            "label": "Description",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(font_family="Source Sans Pro", plot_bgcolor="white", paper_bgcolor="white")
    return fig

display(bar_types(overall, "K1", "K1 engines — registrations by type"))
display(bar_types(overall, "K2", "K2 seats — registrations by type"))
display(bar_types(overall, "K3", "K3 gearboxes — registrations by type"))

body = overall[overall["category"].isin(["K4", "K5", "K6", "K7"])]
fig_body = px.bar(
    body,
    x="component_type",
    y="n_registrations",
    color="label",
    title="Body platforms K4–K7 — registrations",
    labels={
        "component_type": "Body type",
        "n_registrations": "Registered vehicles",
        "label": "Description",
    },
    color_discrete_sequence=PLOTLY_BLUES,
)
fig_body.update_layout(font_family="Source Sans Pro", plot_bgcolor="white", paper_bgcolor="white")
fig_body.show()


In [ ]:
def trend_chart(category):
    yearly = (
        final[final["category"] == category]
        .groupby(["year", "component_type"], as_index=False)["n_registrations"]
        .sum()
    )
    fig = px.line(
        yearly,
        x="year",
        y="n_registrations",
        color="component_type",
        markers=True,
        title=f"Yearly registrations — {category}",
        labels={
            "year": "Registration year",
            "n_registrations": "Registered vehicles",
            "component_type": "Component type",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(
        font_family="Source Sans Pro",
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(dtick=1),
    )
    return fig


for cat in ["K1", "K2", "K3"]:
    display(trend_chart(cat))

body_yearly = (
    final[final["category"].isin(["K4", "K5", "K6", "K7"])]
    .groupby(["year", "category"], as_index=False)["n_registrations"]
    .sum()
)
fig_body_trend = px.line(
    body_yearly,
    x="year",
    y="n_registrations",
    color="category",
    markers=True,
    title="Yearly registrations — body platforms K4–K7",
    labels={
        "year": "Registration year",
        "n_registrations": "Registered vehicles",
        "category": "Body category",
    },
    color_discrete_sequence=PLOTLY_BLUES,
)
fig_body_trend.update_layout(
    font_family="Source Sans Pro",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(dtick=1),
)
fig_body_trend.show()


### Recommendation

From the totals above:

- **K1:** `K1BE1` and `K1DI1` are joint leaders (OEM1 petrol and diesel). There
  is no lasting diesel-vs-petrol fashion; they stay close in every year.
- **K2:** `K2ST1` fabric seats (OEM1). Fabric leads leather every year.
- **K3:** `K3SG1` manual gearbox (OEM1). Manual leads automatic every year.
- **K4–K7:** each category has only one type. Among the four bodies, **K4**
  (OEM1 Typ11) has the most registrations.

**One car to build (feasible mix):** K4 body, K2ST1 seats, K3SG1 gearbox, and
either K1BE1 or K1DI1. Do not fit K4–K7 together.

### Web app

The Streamlit app uses **only** `Data/SoSe26_Case_Study_finalData_Group_38.csv`.

```text
streamlit run SoSe26_Case_Study_App_Group_38.py
```

Design: light blue, Source Sans Pro (files in `www/fonts/`), logo in
`www/img/logo.svg`. Add screenshots of the four tabs to
`Additional_files/` before submission.
